# 06 Video Evaluation: CNN + LSTM

Evaluate the trained EfficientNet-B0 + LSTM video deepfake classifier on the test split.

## Step 1: Setup

This notebook uses the same video preprocessing and CNN+LSTM architecture as `05_train_video_model.ipynb`.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

DATA_ROOT = Path("../data/splits/videos")
MODEL_PATH = Path("../models/video/cnn_lstm_model.pth")

SPLITS = ["train", "val", "test"]
CLASSES = ["real", "fake"]
LABEL_MAP = {"real": 0, "fake": 1}
ID_TO_LABEL = {value: key for key, value in LABEL_MAP.items()}
VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

SEQUENCE_LENGTH = 20
IMAGE_SIZE = 224
BATCH_SIZE = 2
NUM_WORKERS = 0
LSTM_HIDDEN_SIZE = 256
LSTM_NUM_LAYERS = 1
FREEZE_CNN = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Model path exists: {MODEL_PATH.exists()} | {MODEL_PATH}")

## Step 2: Load Test Data

Load the test videos with the same dataset pipeline used during training.

In [ ]:
def build_video_dataframe(data_root: Path) -> pd.DataFrame:
    records = []

    for split in SPLITS:
        for class_name in CLASSES:
            class_dir = data_root / split / class_name
            print(f"Scanning {class_dir} | exists={class_dir.exists()}")

            if not class_dir.exists():
                continue

            for video_path in sorted(class_dir.rglob("*")):
                if video_path.suffix.lower() in VIDEO_EXTENSIONS:
                    records.append(
                        {
                            "video_path": str(video_path),
                            "split": split,
                            "label_name": class_name,
                            "label": LABEL_MAP[class_name],
                        }
                    )

    return pd.DataFrame(records)


df = build_video_dataframe(DATA_ROOT)
print(f"Loaded records: {len(df)}")
df.head()

In [ ]:
if len(df) > 0:
    display(pd.crosstab(df["split"], df["label_name"], margins=True))
else:
    print("No videos found yet. Add files under ../data/splits/videos before evaluation.")

## Same Preprocessing

Frames are sampled to a fixed sequence length, resized to 224 x 224, converted to tensors, and normalized with ImageNet statistics.

In [ ]:
frame_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


def extract_frames(video_path: str, sequence_length: int = SEQUENCE_LENGTH, transform=frame_transform) -> torch.Tensor:
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []

    if total_frames > 0:
        frame_indices = np.linspace(0, max(total_frames - 1, 0), sequence_length).astype(int)
    else:
        frame_indices = np.arange(sequence_length)

    for frame_index in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
        success, frame = cap.read()

        if not success:
            continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = Image.fromarray(frame)
        frames.append(transform(frame))

    cap.release()

    if len(frames) == 0:
        frames = [torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)]

    while len(frames) < sequence_length:
        frames.append(frames[-1].clone())

    frames = frames[:sequence_length]
    return torch.stack(frames)  # (sequence_length, 3, 224, 224)

In [ ]:
class DeepfakeVideoDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, sequence_length: int = SEQUENCE_LENGTH, transform=frame_transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.sequence_length = sequence_length
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        frames = extract_frames(row["video_path"], self.sequence_length, self.transform)
        label = torch.tensor(row["label"], dtype=torch.float32)
        return frames, label, row["video_path"]


test_df = df[df["split"] == "test"].copy() if len(df) > 0 else pd.DataFrame()
test_dataset = DeepfakeVideoDataset(test_df)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Test videos: {len(test_dataset)}")

## Step 1: Load Model

Recreate the CNN+LSTM architecture, load the trained checkpoint, and switch to evaluation mode.

In [ ]:
class CNNLSTMDeepfakeDetector(nn.Module):
    def __init__(
        self,
        hidden_size: int = LSTM_HIDDEN_SIZE,
        num_layers: int = LSTM_NUM_LAYERS,
        freeze_cnn: bool = FREEZE_CNN,
    ):
        super().__init__()

        weights = models.EfficientNet_B0_Weights.DEFAULT
        self.cnn = models.efficientnet_b0(weights=weights)
        self.feature_dim = self.cnn.classifier[1].in_features
        self.cnn.classifier = nn.Identity()
        self.freeze_cnn = freeze_cnn

        if self.freeze_cnn:
            for parameter in self.cnn.parameters():
                parameter.requires_grad = False

        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_size, 1),
        )

    def forward(self, video_batch: torch.Tensor, debug: bool = False):
        batch_size, sequence_length, channels, height, width = video_batch.shape
        frames = video_batch.reshape(batch_size * sequence_length, channels, height, width)

        if self.freeze_cnn:
            with torch.no_grad():
                frame_features = self.cnn(frames)
        else:
            frame_features = self.cnn(frames)

        feature_sequence = frame_features.reshape(batch_size, sequence_length, self.feature_dim)
        lstm_output, _ = self.lstm(feature_sequence)
        final_timestep = lstm_output[:, -1, :]
        logits = self.classifier(final_timestep).squeeze(1)

        if debug:
            print(f"Batch shape (video tensor): {video_batch.shape}")
            print(f"Feature shape after CNN: {frame_features.shape}")
            print(f"Feature sequence shape: {feature_sequence.shape}")
            print(f"LSTM output shape: {lstm_output.shape}")
            print(f"Logits shape: {logits.shape}")

        return logits

In [ ]:
def load_cnn_lstm_model(model_path: Path) -> CNNLSTMDeepfakeDetector:
    checkpoint = torch.load(model_path, map_location=device)

    hidden_size = checkpoint.get("lstm_hidden_size", LSTM_HIDDEN_SIZE)
    num_layers = checkpoint.get("lstm_num_layers", LSTM_NUM_LAYERS)
    model = CNNLSTMDeepfakeDetector(hidden_size=hidden_size, num_layers=num_layers).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model


if MODEL_PATH.exists():
    model = load_cnn_lstm_model(MODEL_PATH)
    print("Loaded CNN+LSTM checkpoint and set model to eval mode.")
else:
    model = CNNLSTMDeepfakeDetector().to(device)
    model.eval()
    print("Checkpoint not found. Initialized model architecture only; run training first for real evaluation.")

## Step 3: Inference

For each video, extract frames, pass the frame sequence through CNN + LSTM, and collect predicted labels and fake probabilities.

In [ ]:
def run_video_inference(model: nn.Module, loader: DataLoader) -> pd.DataFrame:
    model.eval()
    rows = []

    if len(loader.dataset) == 0:
        print("No test videos available for inference yet.")
        return pd.DataFrame(rows)

    with torch.no_grad():
        for videos, labels, paths in tqdm(loader, desc="Evaluating videos"):
            videos = videos.to(device)
            logits = model(videos)
            probabilities = torch.sigmoid(logits).detach().cpu().numpy()
            predictions = (probabilities >= 0.5).astype(int)

            for path, true_label, probability, prediction in zip(paths, labels.numpy(), probabilities, predictions):
                rows.append(
                    {
                        "video_path": path,
                        "true_label": int(true_label),
                        "true_label_name": ID_TO_LABEL[int(true_label)],
                        "fake_probability": float(probability),
                        "predicted_label": int(prediction),
                        "predicted_label_name": ID_TO_LABEL[int(prediction)],
                    }
                )

    return pd.DataFrame(rows)


predictions_df = run_video_inference(model, test_loader)
predictions_df.head()

## Step 4: Metrics

Compute accuracy, precision, recall, and F1 score for binary video classification.

In [ ]:
if len(predictions_df) == 0:
    print("Metrics skipped because no predictions are available.")
    metrics_df = pd.DataFrame()
else:
    y_true = predictions_df["true_label"].values
    y_pred = predictions_df["predicted_label"].values

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
    }
    metrics_df = pd.DataFrame([metrics])

metrics_df

## Step 5: Visualization

Plot the confusion matrix, prediction distribution, and per-video confidence.

In [ ]:
if len(predictions_df) == 0:
    print("Confusion matrix skipped because no predictions are available.")
else:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["real", "fake"])
    display.plot(cmap="Blues", values_format="d")
    plt.title("CNN+LSTM Video Confusion Matrix")
    plt.show()

In [ ]:
if len(predictions_df) == 0:
    print("Prediction distribution skipped because no predictions are available.")
else:
    ax = predictions_df["predicted_label_name"].value_counts().reindex(CLASSES, fill_value=0).plot(
        kind="bar",
        color=["#4c78a8", "#f58518"],
        figsize=(6, 4),
    )
    ax.set_title("Prediction Distribution")
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("Video count")
    plt.xticks(rotation=0)
    plt.show()

In [ ]:
if len(predictions_df) == 0:
    print("Confidence plot skipped because no predictions are available.")
else:
    sorted_predictions = predictions_df.sort_values("fake_probability").reset_index(drop=True)
    colors = np.where(sorted_predictions["predicted_label"].values == 1, "#f58518", "#4c78a8")

    plt.figure(figsize=(10, 4))
    plt.bar(np.arange(len(sorted_predictions)), sorted_predictions["fake_probability"], color=colors)
    plt.axhline(0.5, color="black", linestyle="--", linewidth=1)
    plt.title("Per-Video Fake Confidence")
    plt.xlabel("Videos sorted by fake probability")
    plt.ylabel("Fake probability")
    plt.ylim(0, 1)
    plt.show()

## Step 6: Debugging

Print sample predictions and incorrect predictions to inspect model behavior.

In [ ]:
if len(predictions_df) == 0:
    print("No sample predictions available.")
else:
    display(predictions_df.head(10))

In [ ]:
if len(predictions_df) == 0:
    print("No incorrect predictions available.")
else:
    incorrect_predictions = predictions_df[predictions_df["true_label"] != predictions_df["predicted_label"]].copy()
    print(f"Incorrect predictions: {len(incorrect_predictions)} / {len(predictions_df)}")
    display(incorrect_predictions.sort_values("fake_probability", ascending=False).head(20))

## Step 7: Comparison

Comparison between:
1. CNN+LSTM model
2. Frame-based image model (used in deployment)

Discuss:
- speed
- accuracy
- complexity

### Discussion

The CNN+LSTM model can learn temporal patterns across frames, which may improve accuracy when motion artifacts, flickering, inconsistent facial geometry, or frame-to-frame blending errors are useful signals. Its main cost is speed: every video requires extracting multiple frames, running the CNN across the full sequence, and then running the LSTM. It is also more complex to train, tune, debug, and deploy.

The frame-based image model is simpler and faster because it treats selected frames independently. This makes deployment easier, reduces inference latency, and avoids maintaining a recurrent video pipeline. The tradeoff is that it may miss temporal artifacts that only appear across a sequence of frames.

For this project, the CNN+LSTM model is useful as an academic video-level baseline, while the frame-based image model remains the deployment-oriented approach because it is more efficient and easier to integrate.